In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.getcwd())

In [ ]:
os.chdir("/content/drive/My Drive/베이지안자료분석PBL")

In [ ]:
btc_data = pd.read_csv("./data/raw_data/binance_btc.csv")

btc_data = btc_data[["open_time", "open", "high", "low", "close", "volume"]].rename(
    columns={"open_time" : "date"}
)
btc_data["date"] = pd.to_datetime(btc_data["date"])

btc_data.describe()

In [ ]:
def cal_mfi(df, period=14):
    df['tp'] = (df['high'] + df['low'] + df['close']) / 3
    df['rmf'] = df['tp'] * df['volume']
    df['positive_flow'] = df['rmf'].where(df['tp'] > df['tp'].shift(1), 0)
    df['negative_flow'] = df['rmf'].where(df['tp'] < df['tp'].shift(1), 0)
    df['sum_positive'] = df['positive_flow'].rolling(window=period).sum()
    df['sum_negative'] = df['negative_flow'].rolling(window=period).sum()
    df['mfr'] = df['sum_positive'] / df['sum_negative']
    df['mfi'] = 100 - (100 / (1 + df['mfr']))

    return df['mfi']

In [ ]:
def cal_williams(df, period=14):
    df['highest_high'] = df['high'].rolling(window=period).max()
    df['lowest_low'] = df['low'].rolling(window=period).min()

    df['williams'] = ((df['highest_high'] - df['close']) /
                        (df['highest_high'] - df['lowest_low'])) * -100

    return df['williams']

In [ ]:
def cal_future_close_pct(df):
    df['close_pct'] = (df['close'].shift(-1) - df['close']) / df['close']

    return df['close_pct']

In [ ]:
def cal_past_close_pct(df):
    df['close_pct'] = (df['close'] - df['close'].shift(1)) / df['close'].shift(1)

    return df['close_pct']

In [ ]:
def cal_past_volume_pct(df):
    df['volume_pct'] = (df['volume'] - df['volume'].shift(1)) / df['volume'].shift(1)

    return df['volume_pct']

In [ ]:
def cal_past_open_pct(df):
    df['open_pct'] = (df['open'] - df['open'].shift(1)) / df['open'].shift(1)

    return df['open_pct']

In [ ]:
def cal_past_high_pct(df):
    df['high_pct'] = (df['high'] - df['high'].shift(1)) / df['high'].shift(1)

    return df['high_pct']

In [ ]:
def cal_past_low_pct(df):
    df['low_pct'] = (df['low'] - df['low'].shift(1)) / df['low'].shift(1)

    return df['low_pct']

In [ ]:
df = pd.DataFrame()
df['future_close_pct'] = cal_future_close_pct(btc_data) * 100
df['past_open_pct'] = cal_past_open_pct(btc_data) * 100
df['past_close_pct'] = cal_past_close_pct(btc_data) * 100
df['past_volume_pct'] = cal_past_volume_pct(btc_data) * 100
df['past_high_pct'] = cal_past_high_pct(btc_data) * 100
df['past_low_pct'] = cal_past_low_pct(btc_data) * 100
df['mfi'] = cal_mfi(btc_data)
df['williams'] = cal_williams(btc_data)
df['date'] = btc_data['date']
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'df' is your DataFrame
# Exclude 'future_close_pct' from the independent variables
independent_variables = df.columns.difference(['future_close_pct', 'date'])

# Create scatter plots for each independent variable against 'future_close_pct'
for variable in independent_variables:
    plt.figure(figsize=(6, 4))
    plt.scatter(df[variable], df['future_close_pct'], alpha=0.6)
    plt.title(f'Scatter Plot: {variable} vs future_close_pct')
    plt.xlabel(variable)
    plt.ylabel('future_close_pct')
    plt.grid(True)
    plt.show()


In [ ]:
# df[(df['future_close_pct'] > 0) & (df['future_close_pct'] < 2)]['mfi'].hist(bins=100)

df[(df['future_close_pct'] > 0) & (df['future_close_pct'] < 2)]['temp'].hist(bins=100)
# df[(df['future_close_pct'] > 0.04) & (df['future_close_pct'] < 0.06)]['mfi'].hist()

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'df' is your DataFrame
# Exclude 'future_close_pct' and 'date' from the independent variables
independent_variables = df.columns.difference(['future_close_pct', 'date'])

# Determine the layout for subplots (e.g., 3x3 grid for 9 variables)
num_vars = len(independent_variables)
cols = 3  # Number of columns in the grid
rows = (num_vars + cols - 1) // cols  # Calculate rows to fit all variables

# Create a figure and subplots with reduced size
fig, axes = plt.subplots(rows, cols, figsize=(10, 4 * rows))  # Adjust figure size
axes = axes.flatten()  # Flatten the array for easy iteration

# Plot each variable
for i, variable in enumerate(independent_variables):
    ax = axes[i]
    ax.scatter(df[variable], df['future_close_pct'], alpha=0.6, s=10)  # Smaller points
    ax.set_title(f'{variable} vs future_close_pct', fontsize=10)  # Smaller titles
    ax.set_xlabel(variable, fontsize=8)  # Smaller labels
    ax.set_ylabel('future_close_pct', fontsize=8)  # Smaller labels
    ax.grid(True)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Adjust layout
plt.tight_layout()
plt.show()


In [ ]:
df.isna().sum()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.info()

In [ ]:
df.to_csv("./data/raw_data/btc_data.csv", index=False)

In [ ]:
df

In [ ]:
df.drop('date', axis=1).plot()

In [ ]:
df['past_close_pct'].plot(kind='kde')

In [ ]:
df.columns

In [ ]:
cols = df.columns

X = df[cols].drop(['future_close_pct', 'date'], axis=1).values
y = df['future_close_pct'].values

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_scaled = (y - np.mean(y)) / np.std(y)  # 종속변수도 표준화

In [ ]:
import pymc as pm
import arviz as az

# 베이지안 회귀 모델 정의
with pm.Model() as model:
    # Priors 정의
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_scaled.shape[1])  # 계수들
    intercept = pm.Normal("intercept", mu=0, sigma=1)  # 절편
    sigma = pm.HalfNormal("sigma", sigma=1)  # 오차항

    # 선형 모델
    mu = pm.math.dot(X_scaled, beta) + intercept

    # 관측된 데이터
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_scaled)

    # 샘플링
    trace = pm.sample(2000, tune=1000, return_inferencedata=True, random_seed=42)

# 결과 확인
az.plot_trace(trace)
az.summary(trace, hdi_prob=0.95)

In [ ]:
with model:
    # 새로운 데이터에 대한 예측
    posterior_predictive = pm.sample_posterior_predictive(trace, var_names=["y_obs"])

# 예측 값
predicted_values = posterior_predictive.posterior_predictive["y_obs"].values

# 평균 값 계산
predicted_mean = predicted_values.mean(axis=0)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 원하는 그룹 선택 (첫 번째 그룹)
group_predicted_values = predicted_values[0]  # Shape: (2000, 2618)

# 데이터 축(axis=0) 기준 평균 계산
predicted_mean = group_predicted_values.mean(axis=0)  # Shape: (2618,)

# 평가
rmse = mean_squared_error(y_scaled, predicted_mean, squared=False)
mae = mean_absolute_error(y_scaled, predicted_mean)
r2 = r2_score(y_scaled, predicted_mean)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R²: {r2}")

In [ ]:
print(predicted_values.shape)  # 예상: (2000, 2618)